In [43]:
import os
import pandas as pd
import datetime
from sqlalchemy import *
from sqlalchemy_utils import database_exists, create_database
from sqlalchemy_privileges import *
from sqlalchemy.sql import text, quoted_name

In [ ]:
# Chargement de la base de données de test
data_path = '..\\..\\data\\processed'
test_df = pd.read_csv(os.path.join(data_path, 'test.csv'))

In [ ]:
# Connection à l'utilisateur admin de PostgresSQL
POSTGRES_PASSWORD = "runJNJsCS3a3drSV"

DATABASE_NAME = "scoring_db"
DATABASE_USER = "scoring_app"
DATABASE_USER_PASSWORD = "scoring_app_pswd"

admin_url = URL.create(
    drivername="postgresql+psycopg2",
    username="postgres",
    password=POSTGRES_PASSWORD,
    host="localhost",
    port=5432,
    database="postgres",
)

admin_engine = create_engine(admin_url, isolation_level="AUTOCOMMIT")

In [ ]:
# A travers l'admin, je crée un nouvel utilisateur pour ma base de données de scoring
create_user_sql = text(f"CREATE USER {quoted_name(DATABASE_USER, False)} WITH PASSWORD :database_password")
with admin_engine.connect() as connection:
    connection.execute(create_user_sql,{"database_password": DATABASE_USER_PASSWORD})

In [ ]:
# Je crée ensuite la nouvelle base de donnée et j'y attribue l'utilisateur
with admin_engine.connect() as connection:
    connection.execute(text(f"CREATE DATABASE {quoted_name(DATABASE_NAME, False)} OWNER scoring_app"))

In [ ]:
# Je peux maintement me connecté à la nouvelle base de données
scoring_url = URL.create(
    drivername="postgresql+psycopg2",
    username=DATABASE_USER,
    password=DATABASE_USER_PASSWORD,
    host="localhost",
    port=5432,
    database=DATABASE_NAME,
)

scoring_engine = create_engine(scoring_url, isolation_level='AUTOCOMMIT')

In [ ]:
# J'y ajoute une table avec mes données de tests grâce à la fonction pandas to_sql
test_df.to_sql(name = 'test_clients', con = scoring_engine, if_exists = "replace")

In [ ]:
# Paramétrage de la clé primaire de la table: l'id des clients
with scoring_engine.connect() as connection:
    connection.execute(text("""
        ALTER TABLE test_clients
        ADD CONSTRAINT clients_pkey PRIMARY KEY ("SK_ID_CURR");
    """))

In [ ]:
# Je charge cette table dans un objet MetaData et je crée également dans
# cet objet la table qui sauvegardera les outputs du modèle. 
metadata = MetaData()

test_clients = Table(
    "test_clients",
    metadata,
    autoload_with=scoring_engine,
)

model_logs = Table(
    "model_logs", metadata,
    Column("request_id", BigInteger, Identity(always=True), primary_key=True),
    Column("requested_at", TIMESTAMP, nullable=False, default=datetime.datetime.now(datetime.timezone.utc)),
    Column("client_id", Integer, ForeignKey("test_clients.SK_ID_CURR", ondelete="CASCADE"), nullable=False ),
    Column("pred_class", Integer, nullable=False),
    Column("execution_time_ms", Numeric(10, 3), nullable=False),
)
# Cette commande permet de créer la nouvelle table dans la DB
metadata.create_all(scoring_engine)

In [ ]:
#Visualisation de la structure finale de la DB.
inspector = inspect(scoring_engine)

# Liste des tables
tables = inspector.get_table_names()

print("Tables dans la base :")
for table_name in tables:
    print(f"- {table_name}")

    if table_name == "model_logs":
        columns = inspector.get_columns(table_name)

        for column in columns:
            print(
                f"{column['name']} | "
                f"type={column['type']} | "
                f"nullable={column['nullable']} | "
                f"default={column['default']}"
            )

Tables dans la base :
- test_clients
- model_logs
request_id | type=BIGINT | nullable=False | default=None
requested_at | type=TIMESTAMP | nullable=False | default=None
client_id | type=INTEGER | nullable=False | default=None
pred_class | type=INTEGER | nullable=False | default=None
execution_time_ms | type=NUMERIC(10, 3) | nullable=False | default=None
